## Análise

Nesse notebook é feita a análise dos dados empregada no livro.

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from utils.load_csv import load_csv
from utils.save_csv import save_csv

In [2]:
dados = load_csv('dados.csv')

In [3]:
chi2, p, dof, expected = chi2_contingency(dados)

print("Chi2:", chi2)

print("p-valor:", p)

if p < 0.001:
    print("p < 0.001")
else:
    print(f"p = {p:.3f}")

Chi2: 185.11538691615056
p-valor: 6.348987521003789e-41
p < 0.001


In [4]:
n = dados.to_numpy().sum()

k = min(dados.shape)

cramer_v = np.sqrt(chi2 / (n * (k - 1)))

print("\nV de Cramér:", round(cramer_v, 3))

if cramer_v < 0.1:
    interpretacao = "efeito desprezível"
elif cramer_v < 0.3:
    interpretacao = "efeito pequeno"
elif cramer_v < 0.5:
    interpretacao = "efeito moderado"
else:
    interpretacao = "efeito forte"

print("Interpretação:", interpretacao)


V de Cramér: 0.335
Interpretação: efeito moderado


In [5]:
residuos_padronizados = (dados - expected) / np.sqrt(expected)

residuos_padronizados = pd.DataFrame(residuos_padronizados, 
                        index = dados.index, 
                        columns = dados.columns)

residuos_padronizados.index = ["Lula", "Collor"]

print(residuos_padronizados.round(1))

        Esquerda  Centro  Direita
Lula         7.9     0.9     -6.9
Collor      -6.4    -0.8      5.6


In [6]:
n = dados.to_numpy().sum()
row_totals = dados.sum(axis=1)
col_totals = dados.sum(axis=0)

p_linha = row_totals / n
p_coluna = col_totals / n

expected_df = pd.DataFrame(expected, 
                           index=dados.index, 
                           columns=dados.columns)

residuos_ajustados = (dados - expected_df) / np.sqrt(
    expected_df * 
    (1 - p_linha.values.reshape(-1,1)) *
    (1 - p_coluna.values.reshape(1,-1))
)

residuos_ajustados.index = ["Lula", "Collor"]

print(residuos_ajustados.round(1))

        Esquerda  Centro  Direita
Lula        11.2     1.7    -10.8
Collor     -11.2    -1.7     10.8


In [7]:
save_csv(residuos_ajustados, 'residuos_ajustados.csv')

Base salva em data\residuos_ajustados.csv
